# Hash Generation
Reads every **active** row in `hash_config`, hashes each configured source table, and writes the results to the `hashed_values` output table.

For every configured table:
- **primary_key_hash_value** = hash of the primary key column's value
- **composite_keys_hash_value** = hash of the `||`-concatenated composite key column values
- **hash_value** = hash of the -concatenated values of *every* column in the table, as reported by `information_schema.columns`, **excluding any column whose name ends in `___c`**
- the hash algorithm (`md5` or `sha256`) is whatever `hash_algorithm` says for that table

## Imports

In [0]:
%run ./00_config

## Read active config rows

In [0]:
config_rows = (
    spark.table(config_table)
    .filter(F.col("is_active") == 1)
    .collect()
)

print(f"Found {len(config_rows)} active table(s) to process")

## Helper functions
Standardizes each column's value based on its datatype before concatenating (fixed date/timestamp formats, decimal-safe numerics, trimmed strings, canonical booleans), and makes NULLs explicit rather than letting `concat_ws` silently drop them.

In [0]:
def hash_expr(algorithm, expr):
    """Return a hash column expression for the given algorithm."""
    algorithm = algorithm.strip().lower()
    if algorithm == "md5":
        return F.md5(expr)
    elif algorithm in ("sha256", "sha-256"):
        return F.sha2(expr, 256)
    else:
        raise ValueError(f"Unsupported hash_algorithm: {algorithm}")


def get_hashable_columns(catalog, schema, table):
    cols_df = spark.sql(f"""
        SELECT column_name, data_type
        FROM {catalog}.information_schema.columns
        WHERE table_schema = '{schema}'
          AND table_name   = '{table}'
          AND column_name NOT LIKE '%___c'
        ORDER BY ordinal_position
    """)
    return [(r["column_name"], r["data_type"]) for r in cols_df.collect()]

NULL_PLACEHOLDER = "<NULL>"


def standardize_column(colname, data_type):
    return F.coalesce(
        F.col(colname).cast("string"),
        F.lit(NULL_PLACEHOLDER)
    )


def concat_columns(col_specs):
    return F.concat(
        *[standardize_column(name, dtype) for name, dtype in col_specs]
    )

## Process each configured table, one at a time
Each table is fully processed and **appended to the output table immediately**, before moving on to the next table. This avoids holding every table's results in memory at once, and means a failure partway through only affects the table being processed at that moment.

Before appending, any existing rows for that same `catalog/schema/table` are deleted from the output table, so re-running the notebook refreshes that table's rows instead of duplicating them.

In [0]:
output_table = f"{tgt_catalog}.{tgt_schema}.{tgt_table}"

for row in config_rows:
    src_catalog    = row["catalog"]
    src_schema     = row["schema"]
    src_table      = row["table"]
    primary_key    = row["primary_key"].strip()
    composite_keys = [c.strip() for c in row["composite_keys"].split(",")]
    hash_algorithm = row["hash_algorithm"].strip().lower()

    full_table_name = f"{src_catalog}.{src_schema}.{src_table}"
    print(f"Processing {full_table_name} using {hash_algorithm}...")

    all_columns = get_hashable_columns(src_catalog, src_schema, src_table)
    if not all_columns:
        print(f"  WARNING: no hashable columns found for {full_table_name}, skipping")
        continue

    col_types = dict(all_columns)
    src_df = spark.table(full_table_name)

    pk_value_expr = concat_columns([(primary_key, col_types.get(primary_key, "STRING"))])
    ck_value_expr = concat_columns([(c, col_types.get(c, "STRING")) for c in composite_keys])
    row_value_expr = concat_columns(all_columns)

    result_df = src_df.select(
        F.lit(src_catalog).alias("catalog"),
        F.lit(src_schema).alias("schema"),
        F.lit(src_table).alias("table"),
        pk_value_expr.alias("primary_key"),
        hash_expr(hash_algorithm, pk_value_expr).alias("primary_key_hash_value"),
        ck_value_expr.alias("composite_keys"),
        hash_expr(hash_algorithm, ck_value_expr).alias("composite_keys_hash_value"),
        hash_expr(hash_algorithm, row_value_expr).alias("hash_value"),
        F.lit(hash_algorithm).alias("hash_algo"),
        F.current_timestamp().alias("loaded_at___c"),
    )

    # Refresh (rather than duplicate) this table's rows in the output table
    spark.sql(f"""
        DELETE FROM {output_table}
        WHERE catalog = '{src_catalog}'
          AND schema  = '{src_schema}'
          AND table   = '{src_table}'
    """)

    (
        result_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(output_table)
    )

    row_count = result_df.count()
    print(f"  Appended {row_count} row(s) from {full_table_name} to {output_table}\n")

print("All configured tables processed.")

## Sanity check

In [0]:
display(spark.table(f"{tgt_catalog}.{tgt_schema}.{tgt_table}").limit(50))